In [ ]:
# This cell is labelled 'paramters' to work with papermill remotely
# Papermill will overwite the default local plate value below with whatever is passed to 
# the -p flag in the snakerule shell script
#os.system("conda activate eqtl_study") use this locally if using VScode
plate = 'plate1'
plate = globals().get("plate")
print(f"Processing plate: {plate}")

In [ ]:
# Import custom utility packages, lists and functions
import sys
import os
if os.path.exists('/scratch/'):
    root_dir = '/scratch/c.c1477909/eQTL_study_2025/'
else:
    root_dir = '/Users/darren/Desktop/eQTL_study_2025/'
        
sys.path.append(root_dir + 'workflow/scripts/')

from init_env import *
from anndata_utils import *
from gene_lists import *

# Set variables
resolutions = [0.1] 

In [ ]:
# Initialize the environment and get all paths and logger
logger, root_dir, sheets_dir, plate_path, scanpy_dir = initialize_env(plate)
logger.info("Loading data ...")
adata = sc.read(scanpy_dir + f'adata_clusters.h5ad')

In [ ]:
# Define sex genes
sex_genes = ['XIST', 'RPS4Y1', 'DDX3Y']
available_genes = [gene for gene in sex_genes if gene in adata.var_names]
missing_genes = [gene for gene in sex_genes if gene not in adata.var_names]

# Notify about missing genes
if missing_genes:
    print(f"Warning: The following genes are not in the dataset: {missing_genes}")
if not available_genes:
    raise ValueError("None of the sex genes are present in the dataset!")

# Compute UMAP (optional visualization)
sc.pl.umap(adata, color=available_genes, show=True)  # Set show=True to display, or save with save="umap.png"

# Aggregate by sample (assuming 'sample' column exists in adata.obs)
sex_adata = adata[:, available_genes].to_df()  # Convert to DataFrame
sample_sex_exp = sex_adata.groupby(adata.obs['sample']).mean()  # Mean expression per sample

# Classify with string labels
def classify_sex(row):
    if row['XIST'] > 1.0 and row.get('RPS4Y1', 0) < 0.5:
        return 'Female'
    elif row.get('RPS4Y1', 0) > 0.5 and row['XIST'] < 1.0:
        return 'Male'
    else:
        return 'Ambiguous'

# Add string-based sex column
sample_sex_exp['Predicted_Sex'] = sample_sex_exp.apply(classify_sex, axis=1)

# Add numeric sex column (1 = male, 2 = female, 0 = ambiguous)
def classify_sex_numeric(sex_label):
    if sex_label == 'Male':
        return 1
    elif sex_label == 'Female':
        return 2
    else:
        return 0

sample_sex_exp['Sex_Numeric'] = sample_sex_exp['Predicted_Sex'].apply(classify_sex_numeric)

# Print for verification
print(sample_sex_exp)

# Create a DataFrame with sample ID and both sex columns
output_df = sample_sex_exp[['Predicted_Sex', 'Sex_Numeric']].reset_index()  # 'sample' becomes a column
output_df.columns = ['Sample_ID', 'Predicted_Sex', 'Sex_Numeric']  # Rename columns

# Save to CSV
output_df.to_csv("sex_assignments.csv", index=False)  # No index column in output
print("Saved to sex_assignments.csv")

In [ ]:
# Aggregate expression per donor/sample
sex_gene_exp = adata[:, sex_genes].X.mean(axis=0)  # Averaging across cells
sex_gene_df = pd.DataFrame(sex_gene_exp, index=sex_genes, columns=["Mean Expression"])

print(sex_gene_df)

In [ ]:
logger.info(f"Creating UMAP with cell types ...")
cluster_anns = {
    
    '0': 'RG',
    '1': 'ExN-1',
    '2': 'InN-1',
    '3': 'ExN-2',
    '4': 'InN-2',
    '5': 'Endo-Peri',
    '6': 'MG',
    '7': 'Mig-N',
}

custom_palette = {
    'RG': '#FF5959',        # Red (distinct)
    'ExN-1': '#00B6EB',     # Cyan (light blue)
    'InN-1': '#32CD32',     # Lime Green
    'ExN-2':  '#0061FF',    # Strong Blue (darker)
    'InN-2': '#006400',     # Dark Green (stronger contrast with InN-1)
    'Endo-Peri': '#FF00FF', # Magenta
    'MG': '#FFA500',        # Orange
    'Mig-N': '#FFFF00'      # Yellow
}

# Create a new column in adata.obs with cell type names
adata.obs['cell_type'] = adata.obs['leiden_harmony_0.1'].map(cluster_anns)

In [ ]:
adata

In [ ]:
fig, ax = plt.subplots(figsize=(4.5, 4.5), dpi=300)  # Increase dpi for high resolution

# Black background settings
ax.set_facecolor("black")
fig.patch.set_facecolor("black")

# UMAP Plot with custom colors
sc.pl.umap(
    adata, 
    legend_loc="on data",
    legend_fontsize=13,
    legend_fontoutline=4,
    color="cell_type",  # Change to the desired column in adata.obs
    title='',
    ax=ax,
    show=False,
    palette=custom_palette
)

# Update axis colors to white
ax.spines["bottom"].set_color("white")
ax.spines["left"].set_color("white")
ax.xaxis.label.set_color("white")
ax.yaxis.label.set_color("white")
ax.tick_params(colors="white")

# Save the figure as a high-resolution image
plt.savefig(
    scanpy_dir + "umap_high_res.png",  # File name
    dpi=300,              # DPI for resolution (matches figure DPI)
    bbox_inches="tight",  # Removes extra white space
    facecolor=fig.get_facecolor(),  # Ensures black background is saved
    edgecolor="none",     # No edge color
    format="png"          # File format (can also use "pdf", "jpg", etc.)
)

# Optional: Show plot
plt.show()

# Close the figure to free memory (optional)
plt.close(fig)

In [ ]:
# # Final Markers
# marker_genes_dict = {
#     'RG': ['GLI3', 'SLC1A3'],
#     'ExN-1': ['SATB2'],
#     'InN-1': ['NRXN3', 'CNTNAP2'],
#     'ExN-2': ['GRIK3', 'GAS7'],
#     'InN-2': ['GAD1', 'SOX6', 'DLX6-AS1', 'ERBB4'],
#     'Endo-Peri': ['FN1', 'COL4A1'],
#     'MG': ['C3', 'INPP5D', 'CSF1R'],
#     'Mig-N': ['EBF3']
# }

# custom_palette = [
#     '#FF9999',  # RG
#     '#66CCCC',  # ExN-1
#     '#FFCC99',  # InN-1
#     '#99CCFF',  # ExN-2
#     '#FF99CC',  # InN-2
#     '#CCFF99',  # Endo-Peri
#     '#CCCCFF',  # MG
#     '#FFFF99'   # Mig-N
# ]

# fig, ax = plt.subplots(figsize=(6, 5), dpi=300)  # Adjust figsize as needed

# # Black background settings
# ax.set_facecolor("black")
# fig.patch.set_facecolor("black")

# # Stacked violin plot
# sc.pl.stacked_violin(
#     adata,
#     var_names=marker_genes_dict,  # Your gene dictionary
#     groupby="cell_type",          # Matches the column used in your UMAP
#     ax=ax,
#     show=False,                   # Prevents automatic display
#     palette=custom_palette,       # Reuse your custom UMAP palette
#     standard_scale="var",         # Optional: scales each gene independently
#     dendrogram=False,             # Set to True if you want a dendrogram
#     swap_axes=False               # Set to True if you prefer genes on x-axis
# )

# # Update axis colors to white for contrast
# ax.spines["bottom"].set_color("white")
# ax.spines["top"].set_color("white")
# ax.spines["left"].set_color("white")
# ax.spines["right"].set_color("white")
# ax.xaxis.label.set_color("white")
# ax.yaxis.label.set_color("white")
# ax.tick_params(colors="white", which="both")  # Includes minor ticks
# ax.title.set_color("white")                   # If a title is added later

# # Adjust fonts for publication quality (optional)
# ax.tick_params(labelsize=12)  # Adjust tick label size
# ax.xaxis.label.set_size(14)   # Adjust x-axis label size
# ax.yaxis.label.set_size(14)   # Adjust y-axis label size

# # Show the plot
# plt.tight_layout()  # Ensures proper spacing
# plt.show()